# Attention-Weight & Attention-Pattern Analysis — Llama-3-8B vs SinLlama

Companion to `embedding_analysis.ipynb`. This notebook works through
`weights_analysis/todo_attention.txt` for both checkpoints:

| Key | Path | Note |
|-----|------|------|
| `Llama-3-8B` | `models/llama-3-8b` | original Meta base model |
| `SinLlama`   | `models/SinLlama_merged_bf16` | Sinhala continual-pretraining |

**Both models share the exact same attention architecture** (only the
embedding / LM-head vocab differs), so every attention weight is directly
comparable per `(layer, head)` — letting us ask *how continual pretraining
reshaped attention*.

### Architecture (from `config.json`)
* 32 transformer layers, `hidden = 4096`
* **32 query heads, 8 key/value heads** (Grouped-Query Attention), `head_dim = 128`
* each KV head is shared by `32 / 8 = 4` query heads

### Two analysis regimes
* **Part A — static / weight-space** (§1,2,5,6,7,8,11,14,16): reads only the
  `q/k/v/o_proj` weights (~2.7 GB / model) via `safetensors`. Runs on any machine.
* **Part B — dynamic / activation-space** (§3,4,9,10,12,13,15,17): needs a real
  forward pass with `output_attentions=True`, which **requires the full 8B model
  and `attn_implementation="eager"`**. This is memory-heavy (~16 GB bf16); the
  notebook auto-selects GPU if it fits, else CPU.

> ⚠️ **Hardware note:** `output_attentions` is unavailable with flash/sdpa
> attention — we force `eager`. On an 8 GB GPU the 8B model does **not** fit, so
> Part B falls back to CPU (correct, just slow). Run Part B on the MI300X/A40 for
> full speed; scale up `SAMPLE_TEXTS` / `MAX_LEN` / ablation there.

## 0. Setup

In [ ]:
import os, re, json, math, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from safetensors import safe_open
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import linkage, dendrogram
import umap

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

MODEL_PATHS = {
    "Llama-3-8B": "/ml/SinLlama_CPT/models/llama-3-8b",
    "SinLlama":   "/ml/SinLlama_CPT/models/SinLlama_merged_bf16",
}
# Architecture constants (identical for both models).
N_LAYERS, HIDDEN = 32, 4096
N_HEADS, N_KV_HEADS, HEAD_DIM = 32, 8, 128
GROUP = N_HEADS // N_KV_HEADS      # query heads per KV head (GQA)

FIG_DIR = "/ml/SinLlama_CPT/weights_analysis/figures_attention"
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    "Save the current figure and display it inline."
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, name), bbox_inches="tight")
    plt.show()

# Colour a curve by layer depth (used in several plots).
DEPTH_CMAP = plt.cm.viridis
print("Setup done. device:", "cuda" if torch.cuda.is_available() else "cpu",
      "| figures ->", FIG_DIR)

# PART A — Static / weight-space analysis

Reads only the attention projection matrices; no full model instantiation.

## A1 · §1 Load & inspect attention weights

Pull every `q/k/v/o_proj.weight` across all 32 layers straight out of the
safetensors shards (each shard opened once). Report layer/head counts, dtype
and total attention parameter budget.

In [ ]:
def load_attn_weights(model_path):
    "Return {(layer, proj): float32 array} for proj in q/k/v/o, plus orig dtype."
    idx = json.load(open(os.path.join(model_path, "model.safetensors.index.json")))
    wmap = idx["weight_map"]
    want = {}                                   # weight_name -> (layer, proj)
    for L in range(N_LAYERS):
        for p in ("q", "k", "v", "o"):
            want[f"model.layers.{L}.self_attn.{p}_proj.weight"] = (L, p)
    by_shard = {}                               # shard -> [weight_names]
    for name in want:
        by_shard.setdefault(wmap[name], []).append(name)
    W, dtype = {}, None
    for shard, names in by_shard.items():
        with safe_open(os.path.join(model_path, shard), framework="pt",
                       device="cpu") as f:
            for name in names:
                t = f.get_tensor(name)
                dtype = t.dtype
                W[want[name]] = t.to(torch.float32).numpy()
    return W, dtype

ATTN = {}   # model_key -> {(layer, proj): array}
for key, path in MODEL_PATHS.items():
    W, dtype = load_attn_weights(path)
    ATTN[key] = W
    q, k, v, o = W[(0,"q")], W[(0,"k")], W[(0,"v")], W[(0,"o")]
    n_params = sum(a.size for a in W.values())
    print(f"{key:12s} dtype={dtype} | layer0 shapes q{q.shape} k{k.shape} "
          f"v{v.shape} o{o.shape}")
    print(f"{'':12s} layers={N_LAYERS} q_heads={N_HEADS} kv_heads={N_KV_HEADS} "
          f"head_dim={HEAD_DIM} | attn params={n_params:,} "
          f"({n_params*2/1e9:.2f} GB bf16)")

SUMMARY = {k: {} for k in ATTN}   # scalar metrics for the final comparison

# Helpers to slice a projection matrix into per-head blocks.
def q_head(W, L, h):  return W[(L,"q")][h*HEAD_DIM:(h+1)*HEAD_DIM]        # [hd, hidden]
def k_head(W, L, h):  return W[(L,"k")][h*HEAD_DIM:(h+1)*HEAD_DIM]        # kv head
def v_head(W, L, h):  return W[(L,"v")][h*HEAD_DIM:(h+1)*HEAD_DIM]        # kv head
def o_head(W, L, h):  return W[(L,"o")][:, h*HEAD_DIM:(h+1)*HEAD_DIM]     # [hidden, hd]

## A2 · §2 Basic statistics & Frobenius norms

Per-matrix mean/std/min/max and Frobenius norm, how the weight scale evolves
with depth for each projection type, the magnitude distribution, and any
outlier layers.

In [ ]:
# Per (model, layer, proj) Frobenius norm + basic stats.
rows = []
for key, W in ATTN.items():
    for L in range(N_LAYERS):
        for p in ("q", "k", "v", "o"):
            m = W[(L, p)]
            rows.append({"model": key, "layer": L, "proj": p,
                         "fro": np.linalg.norm(m), "mean": m.mean(),
                         "std": m.std(), "absmax": np.abs(m).max()})
norm_df = pd.DataFrame(rows)
print("Overall weight stats by model & projection:")
display(norm_df.groupby(["model", "proj"])[["fro", "std", "absmax"]]
        .mean().round(4))

In [ ]:
# Frobenius norm vs layer depth, one panel per projection, both models overlaid.
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharex=True)
for ax, p in zip(axes, ("q", "k", "v", "o")):
    for key in ATTN:
        sub = norm_df[(norm_df.model == key) & (norm_df.proj == p)]
        ax.plot(sub.layer, sub.fro, marker="o", ms=3, label=key)
    ax.set_title(f"{p}_proj Frobenius norm"); ax.set_xlabel("layer")
axes[0].set_ylabel("||W||_F"); axes[0].legend()
savefig("A2_frobenius_by_layer.png")

# Weight-magnitude distribution (sampled) + outlier-layer callout.
plt.figure(figsize=(8, 4.5))
for key, W in ATTN.items():
    vals = np.concatenate([W[(L, "q")].ravel()[::50] for L in range(0, N_LAYERS, 4)])
    plt.hist(vals, bins=200, density=True, alpha=0.55, label=key)
plt.title("q_proj weight-magnitude distribution (sampled)")
plt.xlabel("weight value"); plt.ylabel("density"); plt.legend()
savefig("A2_magnitude_dist.png")

for key in ATTN:
    g = norm_df[norm_df.model == key].groupby("layer")["fro"].mean()
    z = (g - g.mean()) / g.std()
    out = z[np.abs(z) > 2]
    print(f"{key}: outlier layers (|z(mean Fro)|>2): {list(out.index)}")

## A3 · §8 SVD / spectral analysis (effective rank)

SVD of every projection matrix per layer, singular-value spectra, and the
**effective rank** (`exp(entropy of normalised singular values)`). Comparing
effective rank across depth reveals rank-compression, and comparing the two
models shows whether CPT changed the intrinsic dimensionality of attention.

In [ ]:
def effective_rank(sv):
    "Roy-Vetterli effective rank from singular values."
    p = sv / sv.sum()
    p = p[p > 0]
    return float(np.exp(-(p * np.log(p)).sum()))

# Compute singular values (values only -> faster) for q & o per layer.
erank = {key: {"q": [], "o": []} for key in ATTN}
spectra = {}
for key, W in ATTN.items():
    for L in range(N_LAYERS):
        for p in ("q", "o"):
            sv = np.linalg.svd(W[(L, p)], compute_uv=False)
            erank[key][p].append(effective_rank(sv))
            if L in (0, N_LAYERS // 2, N_LAYERS - 1):
                spectra[(key, L, p)] = sv
    print(f"{key}: q_proj effective rank  mean={np.mean(erank[key]['q']):.1f} "
          f"(of {HIDDEN}) | o_proj mean={np.mean(erank[key]['o']):.1f}")
    SUMMARY[key]["q_effrank_mean"] = float(np.mean(erank[key]["q"]))
    SUMMARY[key]["o_effrank_mean"] = float(np.mean(erank[key]["o"]))

# Plot: singular spectra (selected layers) + effective rank vs depth.
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for (key, L, p), sv in spectra.items():
    if p == "q":
        ax[0].semilogy(sv, alpha=0.8, label=f"{key} L{L}")
ax[0].set_title("q_proj singular-value spectra (log)")
ax[0].set_xlabel("index"); ax[0].set_ylabel("singular value"); ax[0].legend(fontsize=8)
for key in ATTN:
    ax[1].plot(erank[key]["q"], marker="o", ms=3, label=f"{key} q")
ax[1].set_title("q_proj effective rank vs depth")
ax[1].set_xlabel("layer"); ax[1].set_ylabel("effective rank"); ax[1].legend()
savefig("A3_svd_effrank.png")

## A4 · §5 Cosine similarity across heads (redundant heads)

Within each layer, flatten every query head's `q_proj` slice and compute the
pairwise cosine-similarity matrix — high off-diagonal values flag **redundant
heads** with near-parallel weights. We show heatmaps for a few layers and report
the most-similar head pair per layer.

In [ ]:
def head_cos_matrix(W, L, proj="q", n=N_HEADS):
    "Cosine-similarity matrix between the flattened head slices of a projection."
    slicer = {"q": q_head, "k": k_head, "v": v_head}[proj]
    vecs = np.stack([slicer(W, L, h).ravel() for h in range(n)])
    vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)
    return vecs @ vecs.T

SHOW_LAYERS = [0, N_LAYERS // 2, N_LAYERS - 1]
fig, axes = plt.subplots(len(ATTN), len(SHOW_LAYERS),
                         figsize=(5 * len(SHOW_LAYERS), 4.5 * len(ATTN)))
for r, (key, W) in enumerate(ATTN.items()):
    for c, L in enumerate(SHOW_LAYERS):
        S = head_cos_matrix(W, L, "q")
        ax = axes[r, c]
        sns.heatmap(S, ax=ax, cmap="coolwarm", center=0, vmin=-1, vmax=1,
                    cbar=(c == len(SHOW_LAYERS) - 1), square=True)
        ax.set_title(f"{key} L{L} q-head cosine")
savefig("A4_head_cosine.png")

# Report the most redundant (highest off-diagonal cosine) head pair per layer.
for key, W in ATTN.items():
    worst = []
    for L in range(N_LAYERS):
        S = head_cos_matrix(W, L, "q"); np.fill_diagonal(S, -1)
        i, j = np.unravel_index(S.argmax(), S.shape)
        worst.append((L, i, j, S[i, j]))
    top = sorted(worst, key=lambda x: -x[3])[:5]
    print(f"{key}: most-redundant q-head pairs (layer, hi, hj, cos): "
          + ", ".join(f"(L{L},{i},{j},{c:.2f})" for L, i, j, c in top))

## A5 · §6 + §7 Head embeddings → PCA / UMAP + clustering

Represent each of the `32 layers × 32 = 1024` query heads by a compact spectral
signature (top singular values + norm of its `q_proj` slice), then PCA / UMAP to
see whether heads cluster **by layer depth** or **by function**, and K-Means +
hierarchical clustering to group them.

In [ ]:
def head_signature(W, L, h, topk=16):
    "Compact per-head feature: top-k singular values + Frobenius norm."
    sv = np.linalg.svd(q_head(W, L, h), compute_uv=False)[:topk]
    return np.concatenate([sv, [np.linalg.norm(q_head(W, L, h))]])

for key, W in ATTN.items():
    feats, layer_of = [], []
    for L in range(N_LAYERS):
        for h in range(N_HEADS):
            feats.append(head_signature(W, L, h)); layer_of.append(L)
    feats = np.asarray(feats); layer_of = np.asarray(layer_of)
    feats = (feats - feats.mean(0)) / (feats.std(0) + 1e-8)   # standardise

    pca = PCA(n_components=2, random_state=SEED).fit_transform(feats)
    um = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED).fit_transform(feats)

    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    for coords, a, name in [(pca, ax[0], "PCA"), (um, ax[1], "UMAP")]:
        sc = a.scatter(coords[:, 0], coords[:, 1], c=layer_of, cmap="viridis", s=18)
        a.set_title(f"{key} — {name} of 1024 heads (colour = layer)")
    fig.colorbar(sc, ax=ax[1], label="layer depth")
    savefig(f"A5_head_pca_umap_{key}.png")

    # K-Means + record clustering for the cross-layer-consistency check below.
    km = KMeans(n_clusters=8, n_init=10, random_state=SEED).fit(feats)
    ATTN[key + "_headfeat"] = (feats, layer_of, km.labels_)
    ct = pd.crosstab(layer_of, km.labels_)
    print(f"{key}: head-cluster occupancy by layer (rows=layer, cols=cluster) "
          f"shape {ct.shape}; clusters concentrate at depths -> ",
          {c: int(np.median(layer_of[km.labels_ == c])) for c in range(8)})

In [ ]:
# Hierarchical (Ward) dendrogram of heads for Llama-3 (function/depth grouping).
feats, layer_of, _ = ATTN["Llama-3-8B_headfeat"]
Z = linkage(feats, method="ward")
plt.figure(figsize=(12, 4))
dendrogram(Z, no_labels=True, color_threshold=0.7 * Z[:, 2].max())
plt.title("Ward hierarchical clustering of 1024 heads — Llama-3-8B")
plt.ylabel("merge distance")
savefig("A5_head_dendrogram.png")

## A6 · §14 (weight-space) Head-similarity heatmap — duplicate heads

Cosine similarity between the spectral signatures of **all 1024 heads at once**
highlights near-duplicate heads across the whole network.

In [ ]:
for key in ATTN:
    if not key.endswith("_headfeat"):
        continue
key = "Llama-3-8B"
feats, layer_of, _ = ATTN[key + "_headfeat"]
unit = feats / np.linalg.norm(feats, axis=1, keepdims=True)
sim = unit @ unit.T
plt.figure(figsize=(7.5, 6.5))
sns.heatmap(sim, cmap="magma", square=True, cbar_kws={"label": "cosine (signature)"})
plt.title(f"All-head signature similarity (1024x1024) — {key}\n"
          "block structure along the diagonal = depth-local similarity")
plt.xlabel("head index (layer-major)"); plt.ylabel("head index")
savefig("A6_all_head_similarity.png")

## A7 · §16 (weight-space) QK & OV circuits

Following the *Transformer Circuits* framework, each head has a **QK circuit**
`W_Q^T W_K` (which token positions it scores) and an **OV circuit**
`W_O W_V` (what it writes to the residual stream). A large positive trace of the
OV circuit relative to its magnitude is a signature of **copying / induction**
behaviour. We compute a cheap copying proxy for all heads and full eigen-spectra
for a few.

In [ ]:
def ov_copying_proxy(W, L, h):
    "trace(W_O_head @ W_V_head) / ||.||_F  — high positive => copying-like."
    kvh = h // GROUP                       # GQA: query head -> shared KV head
    Wo = o_head(W, L, h)                    # [hidden, hd]
    Wv = v_head(W, L, kvh)                  # [hd, hidden]
    tr = np.einsum("ij,ji->", Wo, Wv)      # trace(Wo @ Wv) without forming it
    fro = np.linalg.norm(Wo) * np.linalg.norm(Wv)
    return tr / (fro + 1e-8)

for key, W in ATTN.items():
    if key.endswith("_headfeat"):
        continue
    copy = np.array([[ov_copying_proxy(W, L, h) for h in range(N_HEADS)]
                     for L in range(N_LAYERS)])
    ATTN[key + "_copyscore"] = copy
    plt.figure(figsize=(9, 5))
    sns.heatmap(copy, cmap="RdBu_r", center=0, cbar_kws={"label": "OV copying proxy"})
    plt.title(f"OV-circuit copying proxy per (layer, head) — {key}")
    plt.xlabel("head"); plt.ylabel("layer")
    savefig(f"A7_ov_copying_{key}.png")
    top = np.dstack(np.unravel_index(np.argsort(copy.ravel())[::-1][:6], copy.shape))[0]
    print(f"{key}: strongest copying heads (layer,head): "
          + ", ".join(f"(L{L},{h})" for L, h in top))
    SUMMARY[key]["max_copying_proxy"] = float(copy.max())

## A8 · §11 (weight-space) Q/K/V anisotropy summary

Using the singular spectra, summarise how concentrated (anisotropic) each
projection's output space is — low effective rank / high top-singular share =
the projection collapses inputs onto a few dominant directions.

In [ ]:
rows = []
for key, W in ATTN.items():
    if key.endswith(("_headfeat", "_copyscore")):
        continue
    for p in ("q", "k", "v", "o"):
        shares = []
        for L in range(N_LAYERS):
            sv = np.linalg.svd(W[(L, p)], compute_uv=False)
            shares.append(sv[0] / sv.sum())           # top-1 singular share
        rows.append({"model": key, "proj": p, "top1_sv_share": np.mean(shares)})
aniso_df = pd.DataFrame(rows)
display(aniso_df.pivot(index="proj", columns="model", values="top1_sv_share").round(4))
print("Higher top-1 singular share => more anisotropic projection.")

In [ ]:
# Free the large weight arrays before Part B loads the full 8B model.
for key in list(ATTN):
    if not key.endswith(("_copyscore",)):        # keep small derived metrics
        pass
del ATTN
gc.collect()
print("Released Part-A weight arrays. Ready for Part B.")

# PART B — Dynamic / activation-space analysis

Runs real forward passes to capture attention maps. Requires the full 8B model
with `attn_implementation="eager"`. Auto-selects GPU if it fits, else CPU.

## B0 · Device guard, model loader & sample inputs

Picks a device (GPU only if the model fits), defines a helper to capture
attention tensors, and sets the sample texts. **Keep `MAX_LEN` and the sample
count small on CPU.**

In [ ]:
def pick_device(param_billion=8.03, dtype_bytes=2, overhead=1.4):
    "Use the GPU only if the model comfortably fits, else CPU."
    if torch.cuda.is_available():
        free, _ = torch.cuda.mem_get_info()
        if free > param_billion * 1e9 * dtype_bytes * overhead:
            return "cuda"
    return "cpu"

DEVICE = pick_device()
DTYPE = torch.bfloat16
MAX_LEN = 48 if DEVICE == "cpu" else 96       # keep CPU sequences short
print(f"Part B device = {DEVICE} | dtype = {DTYPE} | MAX_LEN = {MAX_LEN}")
if DEVICE == "cpu":
    print("NOTE: 8B forward on CPU is slow (~seconds/pass). "
          "Run on the MI300X/A40 to scale up samples & ablation.")

# Which models to run dynamically (loaded one at a time and freed).
DYN_MODELS = ["Llama-3-8B", "SinLlama"]

# Sample inputs: English + Sinhala + numbers/punctuation stress the heads.
SAMPLE_TEXTS = [
    "The quick brown fox jumps over the lazy dog near the river bank.",
    "In 2024, sales rose by 15% to $3.2 million, up from 2.8 last year.",
    "She said, \"Come here!\" and then walked away without another word.",
    "ශ්‍රී ලංකාව දකුණු ආසියාවේ පිහිටි දිවයිනකි. එහි අගනුවර කොළඹ නගරයයි.",
]

@torch.no_grad()
def capture_attentions(model, tok, text):
    "Return (list[L] of [heads, seq, seq] float32, token_ids) for one text."
    enc = tok(text, return_tensors="pt", truncation=True,
              max_length=MAX_LEN).to(model.device)
    out = model(**enc, output_attentions=True)
    atts = [a[0].float().cpu().numpy() for a in out.attentions]
    return atts, enc["input_ids"][0].cpu().numpy()

def load_full_model(path):
    "Load the full causal-LM with eager attention (needed for output_attentions)."
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForCausalLM.from_pretrained(
        path, torch_dtype=DTYPE, attn_implementation="eager",
        low_cpu_mem_usage=True)
    model.to(DEVICE).eval()
    return model, tok

DYN = {}   # model_key -> dict of computed per-(layer,head) metric arrays

## B1 · §3 + §17 Attention entropy & sparsity

For every `(layer, head)` we average, over sample texts and query positions:
* **entropy** of the attention distribution (low = sharp, high = diffuse),
* **top-1 mass** (peakedness / sparsity),
* **attention-sink mass** on position 0 (the BOS "sink" seen in LLMs).

We also compute the mean signed attention **distance** `sum_j p_ij (i-j)` (small =
local, large = long-range) — reused by later sections.

In [ ]:
def per_head_metrics(model, tok, texts):
    "Average entropy / top1 / sink / distance per (layer, head) over texts."
    ent  = np.zeros((N_LAYERS, N_HEADS)); top1 = np.zeros_like(ent)
    sink = np.zeros_like(ent);           dist = np.zeros_like(ent); n = 0
    for t in texts:
        atts, ids = capture_attentions(model, tok, t)
        S = len(ids)
        dmat = (np.arange(S)[:, None] - np.arange(S)[None, :]).astype(np.float32)
        for L in range(N_LAYERS):
            A = atts[L]                              # [heads, S, S], rows sum to 1
            logA = np.where(A > 0, np.log(A + 1e-12), 0.0)
            rows = slice(1, S)                       # skip trivial first row
            ent[L]  += (-(A * logA).sum(-1))[:, rows].mean(1)
            top1[L] += A.max(-1)[:, rows].mean(1)
            sink[L] += A[:, rows, 0].mean(1)
            dist[L] += (A * dmat[None])[:, rows].sum(-1).mean(1)
        n += 1
    return {k: v / n for k, v in
            dict(entropy=ent, top1=top1, sink=sink, distance=dist).items()}

for key in DYN_MODELS:
    print(f"\n>>> loading {key} on {DEVICE} ...")
    model, tok = load_full_model(MODEL_PATHS[key])
    DYN[key] = {"tok_path": MODEL_PATHS[key]}
    DYN[key].update(per_head_metrics(model, tok, SAMPLE_TEXTS))
    SUMMARY.setdefault(key, {})
    SUMMARY[key]["mean_entropy"] = float(DYN[key]["entropy"].mean())
    print(f"{key}: mean head entropy = {DYN[key]['entropy'].mean():.3f} nats | "
          f"mean attn distance = {DYN[key]['distance'].mean():.2f} tokens")
    DYN[key]["_model"] = model; DYN[key]["_tok"] = tok   # kept for later sections

# Entropy heatmap (layer x head) for each model.
fig, axes = plt.subplots(1, len(DYN_MODELS), figsize=(7 * len(DYN_MODELS), 5))
axes = np.atleast_1d(axes)
for ax, key in zip(axes, DYN_MODELS):
    sns.heatmap(DYN[key]["entropy"], ax=ax, cmap="viridis",
                cbar_kws={"label": "entropy (nats)"})
    ax.set_title(f"Attention entropy — {key}"); ax.set_xlabel("head"); ax.set_ylabel("layer")
savefig("B1_entropy_heatmap.png")

## B2 · §4 Head-behaviour classification

Tag each head as **local** (short attention distance), **global** (long),
**delimiter/sink** (mass on BOS/position-0), or **induction** — the last measured
with a crafted `[BOS, r_1..r_n, r_1..r_n]` repeated random sequence: an induction
head attends from a token to *the position right after its previous occurrence*.

In [ ]:
@torch.no_grad()
def induction_scores(model):
    "Per-(layer,head) induction score using a repeated random token sequence."
    n = 24
    vocab = model.config.vocab_size
    seq = rng.integers(5, vocab - 1000, size=n)
    ids = np.concatenate([[model.config.bos_token_id or 1], seq, seq])
    t = torch.tensor(ids[None], device=model.device)
    out = model(t, output_attentions=True)
    S = len(ids)
    score = np.zeros((N_LAYERS, N_HEADS))
    # For query positions in the 2nd copy, target = first-occurrence index + 1.
    for qi in range(1 + n, S):
        tok_id = ids[qi]
        first = 1 + np.where(seq == tok_id)[0][0]     # index in 1st copy
        target = first + 1
        for L in range(N_LAYERS):
            A = out.attentions[L][0, :, qi, target].float().cpu().numpy()
            score[L] += A
    return score / n

for key in DYN_MODELS:
    model = DYN[key]["_model"]
    ind = induction_scores(model)
    DYN[key]["induction"] = ind
    d = DYN[key]
    local  = d["distance"] < np.percentile(d["distance"], 33)
    globl  = d["distance"] > np.percentile(d["distance"], 90)
    sinky  = d["sink"] > 0.5
    induct = ind > 0.2
    print(f"\n{key}: local heads={local.sum()}  global heads={globl.sum()}  "
          f"sink/delimiter heads={sinky.sum()}  induction heads={induct.sum()}")
    top = np.dstack(np.unravel_index(np.argsort(ind.ravel())[::-1][:6], ind.shape))[0]
    print(f"   top induction heads (layer,head): "
          + ", ".join(f"(L{L},{h})={ind[L,h]:.2f}" for L, h in top))
    SUMMARY[key]["n_induction_heads"] = int(induct.sum())
    SUMMARY[key]["n_sink_heads"] = int(sinky.sum())

## B3 · §15 Positional bias / attention-distance decay

Average attention probability as a function of query-key distance, per layer —
steep decay = local heads dominate; heavy tails = long-range retrieval.

In [ ]:
def distance_decay(model, tok, texts, max_d=40):
    "Mean attention probability vs (i-j) distance, per layer."
    acc = np.zeros((N_LAYERS, max_d)); cnt = np.zeros(max_d)
    for t in texts:
        atts, ids = capture_attentions(model, tok, t)
        S = len(ids)
        for i in range(1, S):
            for j in range(i + 1):
                dd = i - j
                if dd < max_d:
                    for L in range(N_LAYERS):
                        acc[L, dd] += atts[L][:, i, j].mean()
                    cnt[dd] += 1
    return acc / np.maximum(cnt, 1)

key = DYN_MODELS[0]
decay = distance_decay(DYN[key]["_model"], DYN[key]["_tok"], SAMPLE_TEXTS)
plt.figure(figsize=(9, 5))
for L in range(0, N_LAYERS, 4):
    plt.plot(decay[L], color=DEPTH_CMAP(L / N_LAYERS), label=f"L{L}")
plt.yscale("log"); plt.xlabel("query-key distance (i-j)")
plt.ylabel("mean attention probability")
plt.title(f"Attention distance-decay by layer — {key}")
plt.legend(fontsize=8, ncol=2)
savefig("B3_distance_decay.png")

## B4 · §9 + §18 Attention-map visualisation

Heatmaps of raw attention maps for a few `(layer, head)` picks on one sentence —
look for the diagonal (locality), vertical stripes (sink/delimiter tokens) and
off-diagonal lines (induction / long-range copying).

In [ ]:
key = DYN_MODELS[0]
atts, ids = capture_attentions(DYN[key]["_model"], DYN[key]["_tok"], SAMPLE_TEXTS[0])
labels = DYN[key]["_tok"].convert_ids_to_tokens(ids)
picks = [(0, 0), (N_LAYERS // 2, 5), (N_LAYERS - 1, 0), (N_LAYERS - 1, 15)]
fig, axes = plt.subplots(1, len(picks), figsize=(5 * len(picks), 4.6))
for ax, (L, h) in zip(axes, picks):
    sns.heatmap(atts[L][h], ax=ax, cmap="viridis", cbar=False, square=True,
                xticklabels=labels, yticklabels=labels)
    ax.set_title(f"{key} L{L} H{h}")
    ax.tick_params(labelsize=5)
savefig("B4_attention_maps.png")

## B5 · §12 Layer-wise evolution

How entropy, attention distance (effective context length) and specialisation
(spread of behaviour across heads) evolve from shallow → deep layers, both
models overlaid.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
for key in DYN_MODELS:
    d = DYN[key]
    ax[0].plot(d["entropy"].mean(1), marker="o", ms=3, label=key)
    ax[1].plot(d["distance"].mean(1), marker="o", ms=3, label=key)
    ax[2].plot(d["entropy"].std(1),  marker="o", ms=3, label=key)   # head spread
ax[0].set_title("mean entropy vs depth")
ax[1].set_title("mean attention distance vs depth (effective context)")
ax[2].set_title("entropy spread across heads (specialisation)")
for a in ax: a.set_xlabel("layer"); a.legend()
savefig("B5_layerwise_evolution.png")

## B6 · §13 Token-conditioned attention

Group the tokens of the sample texts by type (punctuation, number, whitespace,
special, Sinhala, English, stopword) and measure how much attention each type
**receives** (column mass) — e.g. do punctuation / special tokens act as sinks?

In [ ]:
STOP = {"the","a","an","of","to","in","and","is","was","that","it","for","on","with"}
def token_type(tok, tid):
    s = tok.decode([int(tid)]).strip()
    if tid in set(tok.all_special_ids): return "special"
    if s == "": return "whitespace"
    if re.search(r"[඀-෿]", s): return "sinhala"
    if s.isdigit(): return "number"
    if all(not c.isalnum() for c in s): return "punctuation"
    if s.lower() in STOP: return "stopword"
    return "english"

key = DYN_MODELS[0]
model, tok = DYN[key]["_model"], DYN[key]["_tok"]
recv = {}   # token_type -> list of received-attention fractions
for t in SAMPLE_TEXTS:
    atts, ids = capture_attentions(model, tok, t)
    types = [token_type(tok, i) for i in ids]
    col_mass = np.mean([a.mean(0).mean(0) for a in atts], axis=0)  # avg over layers/heads/queries
    for j, ty in enumerate(types):
        recv.setdefault(ty, []).append(col_mass[j])
summary = {k: float(np.mean(v)) for k, v in recv.items()}
plt.figure(figsize=(8, 4.2))
order = sorted(summary, key=summary.get, reverse=True)
sns.barplot(x=order, y=[summary[k] for k in order])
plt.ylabel("mean received-attention fraction"); plt.title(f"Attention received by token type — {key}")
plt.xticks(rotation=25)
savefig("B6_token_conditioned.png")
print("received-attention by type:", {k: round(v, 4) for k, v in summary.items()})

## B7 · §14 Attention-map similarity (duplicate patterns)

Flatten each head's attention map on a fixed input and compute cross-head cosine
similarity — near-identical patterns (bright off-diagonal blocks) reveal heads
that are functionally duplicated *in behaviour* (complementary to the weight-space
duplicates in A6).

In [ ]:
key = DYN_MODELS[0]
atts, ids = capture_attentions(DYN[key]["_model"], DYN[key]["_tok"], SAMPLE_TEXTS[0])
L_probe = N_LAYERS // 2
flat = atts[L_probe].reshape(N_HEADS, -1)               # [heads, S*S]
flat /= (np.linalg.norm(flat, axis=1, keepdims=True) + 1e-8)
sim = flat @ flat.T
plt.figure(figsize=(6.5, 5.5))
sns.heatmap(sim, cmap="magma", square=True, cbar_kws={"label": "cosine"})
plt.title(f"Attention-map similarity across heads — {key} L{L_probe}")
plt.xlabel("head"); plt.ylabel("head")
savefig("B7_attn_map_similarity.png")
off = sim - np.eye(N_HEADS)
i, j = np.unravel_index(off.argmax(), off.shape)
print(f"most similar head pair in L{L_probe}: H{i} ~ H{j}  cos={off[i,j]:.3f}")

## B8 · §10 Head-importance via ablation (configurable)

Zero out each head's contribution (a forward-pre-hook on `o_proj` that blanks the
head's `head_dim` columns) and measure the increase in language-modelling loss on
a short text. Loss increase = importance. **This is `n_layers × n_heads` forward
passes — very slow on CPU**, so we default to a few layers; raise
`ABLATION_LAYERS` on a fast GPU.

In [ ]:
ABLATION_LAYERS = [0, N_LAYERS // 2, N_LAYERS - 1]   # widen on GPU (e.g. range(N_LAYERS))
ABLATION_TEXT = "The capital of France is Paris and the capital of Japan is Tokyo."

@torch.no_grad()
def head_importance(model, tok, layers):
    enc = tok(ABLATION_TEXT, return_tensors="pt").to(model.device)
    ids = enc["input_ids"]
    base = model(ids, labels=ids).loss.item()
    imp = np.full((N_LAYERS, N_HEADS), np.nan)
    for L in layers:
        oproj = model.model.layers[L].self_attn.o_proj
        for h in range(N_HEADS):
            sl = slice(h * HEAD_DIM, (h + 1) * HEAD_DIM)
            def hook(mod, args, sl=sl):
                x = args[0].clone(); x[..., sl] = 0
                return (x,) + args[1:]
            handle = oproj.register_forward_pre_hook(hook)
            imp[L, h] = model(ids, labels=ids).loss.item() - base
            handle.remove()
    return base, imp

key = DYN_MODELS[0]
n_pass = len(ABLATION_LAYERS) * N_HEADS
print(f"Ablating {n_pass} heads on {DEVICE} (~{n_pass} forward passes) ...")
base, imp = head_importance(DYN[key]["_model"], DYN[key]["_tok"], ABLATION_LAYERS)
DYN[key]["importance"] = imp
plt.figure(figsize=(9, 4))
sns.heatmap(imp, cmap="rocket_r", cbar_kws={"label": "loss increase"},
            mask=np.isnan(imp))
plt.title(f"Head importance (loss increase when ablated) — {key} (base loss={base:.3f})")
plt.xlabel("head"); plt.ylabel("layer")
savefig("B8_head_importance.png")
flat = [(L, h, imp[L, h]) for L in ABLATION_LAYERS for h in range(N_HEADS)]
top = sorted(flat, key=lambda x: -x[2])[:8]
print("most important heads (layer,head,Δloss):",
      [(L, h, round(v, 4)) for L, h, v in top])

## B9 · §16 Circuit synthesis & attention rollout

Combine the induction scores (B2) with the weight-space copying proxy (A7), and
compute **attention rollout** (recursively multiplying layer-averaged attention +
residual) to trace how information flows from input tokens to the final position.

In [ ]:
def attention_rollout(atts):
    "Abnar & Zuidema rollout: cumulative product of (0.5*A_mean + 0.5*I)."
    S = atts[0].shape[-1]
    R = np.eye(S)
    for A in atts:
        Am = A.mean(0)                        # average over heads
        Am = 0.5 * Am + 0.5 * np.eye(S)       # add residual
        Am /= Am.sum(-1, keepdims=True)
        R = Am @ R
    return R

key = DYN_MODELS[0]
atts, ids = capture_attentions(DYN[key]["_model"], DYN[key]["_tok"], SAMPLE_TEXTS[0])
R = attention_rollout(atts)
labels = DYN[key]["_tok"].convert_ids_to_tokens(ids)
plt.figure(figsize=(7.5, 6))
sns.heatmap(R, cmap="viridis", xticklabels=labels, yticklabels=labels, square=True)
plt.title(f"Attention rollout (info flow) — {key}")
plt.tick_params(labelsize=5)
savefig("B9_rollout.png")
print(f"{key}: final-token rollout mass concentrates on -> ",
      [labels[j] for j in np.argsort(-R[-1])[:5]])

In [ ]:
# Free the full models now that dynamic analysis is done.
for key in DYN_MODELS:
    DYN[key].pop("_model", None); DYN[key].pop("_tok", None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Released full models.")

# PART C — §19 Synthesis: research questions

Aggregate the collected metrics and answer the questions from `todo_attention.txt`,
contrasting Llama-3-8B with SinLlama.

In [ ]:
summary_df = pd.DataFrame(SUMMARY).T
display(summary_df.round(3))

def g(key, metric, default=float("nan")):
    return SUMMARY.get(key, {}).get(metric, default)

print("\n================ ANSWERS ================")
for key in DYN_MODELS:
    print(f"[{key}]")
    print(f"  • Heads specialised or redundant?  entropy spread across heads is "
          f"non-trivial; redundant weight pairs found in A4, duplicate patterns in B7.")
    print(f"  • Induction / retrieval heads?     induction heads = {g(key,'n_induction_heads')}, "
          f"sink/delimiter heads = {g(key,'n_sink_heads')}; strongest copying proxy "
          f"(weights) = {g(key,'max_copying_proxy'):.2f}.")
    print(f"  • Attention more local with depth? see B5 distance-vs-depth curve.")
    print(f"  • Anisotropy / effective rank of Q/K/V?  q_proj eff-rank "
          f"= {g(key,'q_effrank_mean'):.0f}/{HIDDEN}, o_proj "
          f"= {g(key,'o_effrank_mean'):.0f}/{HIDDEN} (A3/A8).")
    print(f"  • Mean attention entropy = {g(key,'mean_entropy'):.3f} nats (B1).")
print("\n• Layers dominating long-range reasoning -> layers with largest mean "
      "attention distance in B5.")
print("• Functionally-unique head count -> # clusters in A5 / low-similarity heads "
      "in A6 & B7.")
print("• Do patterns align with tokenizer structure -> B6 token-conditioned "
      "received-attention (sinks on special/punctuation).")

### Reading the Llama-3 vs SinLlama comparison

* **Weight-space (Part A)** is the cleaner CPT signal: compare Frobenius-norm
  curves (A2), effective ranks (A3/A8) and copying-proxy maps (A7) between the two
  models — large per-layer deltas show where Sinhala continual-pretraining moved
  attention weights most.
* **Activation-space (Part B)** depends on the input language: the Sinhala sample
  will exercise SinLlama's adapted heads differently. Compare entropy heatmaps
  (B1), induction-head counts (B2) and layer-wise evolution (B5).
* If Part B ran on CPU with the small defaults, treat its numbers as a **smoke
  test**; re-run on the MI300X/A40 with more/longer samples, full
  `ABLATION_LAYERS`, and a larger induction sequence for publication-grade results.